# Prompting 101 — Hands-On Practice
**ComplianceGPT Lab · REU 2026**

Companion notebook to `slides-prompting-101.html`. Everything here is **generic** — no HIPAA, no ComplianceGPT code, no repo dependency beyond a running local Ollama. If you've never used ChatGPT/Claude for anything more deliberate than a casual question, start here.

By the end you will have run, with your own eyes: zero-shot, few-shot, chain-of-thought, and reflection (self-critique) prompting, plus four concrete ways prompts fail.

**What's next after this:** `slides-week3.html` applies these exact techniques to a real, messy HIPAA extraction case; `slides-agentic-systems.html` scales reflection up into a real multi-pass research pipeline (with real accuracy numbers, previewed at the very end of this notebook).

In [ ]:
# Run this cell first — checks Ollama is running and picks a model
import requests, json, re

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.2"   # swap for gemma2:2b, phi3:mini, qwen2.5:7b, whatever you have pulled

try:
    tags = requests.get("http://localhost:11434/api/tags", timeout=3).json()
    names = [m["name"] for m in tags["models"]]
    print("Ollama is running \u2713")
    print(f"{MODEL} available:", "\u2713" if any(MODEL in n for n in names) else f"\u2717 \u2014 run: ollama pull {MODEL}")
    print("All models you have:", names)
except Exception as e:
    print("Ollama is NOT reachable. Start it with the Ollama app, or run `ollama serve` in a terminal.")
    print("Error:", e)

In [ ]:
# Your reusable LLM-calling function — you'll use this in every exercise below
def ask(prompt, temperature=0.0, model=MODEL):
    """Send a prompt to a local Ollama model and return the text response."""
    r = requests.post(OLLAMA_URL, json={
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature},
    }, timeout=120)
    r.raise_for_status()
    return r.json()["response"]

def show(label, text):
    print(f"--- {label} " + "-" * max(0, 50 - len(label)))
    print(text.strip())
    print()

---
## Part 1 — What Is a Prompt? (Vague vs. Specific)

**The concept:** the model only has what you typed. No unstated intent, no assumed context. Same request, two prompts — watch the difference.

In [ ]:
vague_prompt = "Write something about dogs."

specific_prompt = """Write a 3-sentence description of golden retrievers \
for a children's pet encyclopedia. Cheerful tone. \
Mention temperament and one fun fact. No jargon."""

show("VAGUE", ask(vague_prompt))
show("SPECIFIC", ask(specific_prompt))

**Look at the two outputs.** The vague one is fine but generic — you'll probably want to rewrite it. The specific one gave you task + audience + length + tone + content requirement + constraint, all explicit. This is the whole game: **put your intent on the page, don't make the model guess it.**

---
## Part 2 — Zero-Shot Prompting

**The concept:** instructions only, no worked examples. Fast to write. Works well for familiar, unambiguous tasks — struggles when "correct" depends on a judgment call you haven't defined.

In [ ]:
zero_shot_prompt = """Classify the sentiment of this review as \
Positive, Negative, or Neutral. Respond with one word only.

Review: "The battery life is incredible but the camera is a huge letdown."""

show("Zero-shot", ask(zero_shot_prompt))

**Where zero-shot breaks:** that review is genuinely mixed — is it Positive, Negative, or Neutral? You never told the model how *you* want mixed reviews handled, so it's guessing your judgment call. Run the cell above a few times, or try `temperature=0.9`, and see if the answer is stable. This is exactly the gap few-shot prompting fixes — next section.

---
## Part 3 — Few-Shot Prompting

**The concept:** show 2–5 worked examples before the real question. The model pattern-matches your examples instead of guessing your intent — usually the single biggest lever for consistent output.

In [ ]:
few_shot_prompt = """Classify each support ticket's urgency as Low, Medium, or High.

Ticket: "App crashes when I open settings."
Urgency: High

Ticket: "Would love a dark mode option someday."
Urgency: Low

Ticket: "Payment went through twice, need a refund."
Urgency:"""

show("Few-shot", ask(few_shot_prompt))

**Your turn (TODO):** write your own 2 examples for a different generic task — e.g. classify a one-line movie review as `Recommend` / `Don't Recommend`, or classify an email opener's tone as `Urgent` / `Casual` / `Formal`. Fill in the blanks below.

In [ ]:
# TODO: replace the ... below with your own task, 2 examples, and a new item to classify
my_few_shot_prompt = """...

Example 1: ...
Label: ...

Example 2: ...
Label: ...

Now classify: ...
Label:"""

# show("My few-shot", ask(my_few_shot_prompt))   # uncomment once you've filled in the prompt above

---
## Part 4 — Chain-of-Thought Prompting

**The concept:** ask the model to reason step-by-step *before* answering. Costs more tokens, but catches errors that come from jumping straight to a conclusion — especially on multi-step logic.

In [ ]:
riddle = """A farmer has 17 sheep. All but 9 die. How many sheep are left?"""

direct_prompt = riddle + "\nAnswer with just the number."
cot_prompt     = riddle + "\nThink step by step, then give the final answer."

show("Direct answer", ask(direct_prompt))
show("Chain-of-thought", ask(cot_prompt))

**The trick in this riddle:** "all but 9 die" means 9 are *left* — the answer is 9, not 8 (17−9). Small/fast models often get the direct version wrong by pattern-matching "looks like subtraction" instead of reading carefully. Chain-of-thought forces the model to state its reasoning where you (and it) can check it.

**Your turn (TODO):** find or write your own logic/math riddle with a similar "trick," and test it both ways below.

In [ ]:
# TODO: put your own riddle here, then compare direct vs. chain-of-thought
my_riddle = "..."

# show("My direct", ask(my_riddle + "\nAnswer with just the number."))
# show("My CoT", ask(my_riddle + "\nThink step by step, then give the final answer."))

---
## Part 5 — Reflection (Self-Critique)

**The concept:** after getting an answer, send it *back* to the model and ask it to check its own work before you accept it. Two calls instead of one — the second call has the first call's answer as context, and a specific instruction to look for errors.

In [ ]:
task = """List 5 U.S. state capitals, paired with their state. \
Format: State - Capital"""

first_answer = ask(task)
show("Pass 1 — initial answer", first_answer)

reflect_prompt = f"""You previously answered this question:
"{task}"

Your answer was:
{first_answer}

Double-check every state-capital pair above for accuracy. \
If anything is wrong, give a corrected full list. \
If everything is correct, say so and repeat the list."""

second_answer = ask(reflect_prompt)
show("Pass 2 — after reflection", second_answer)

**Compare the two passes by hand** — did reflection catch and fix a wrong capital, or did it just repeat pass 1 unchanged? Both outcomes are informative: reflection isn't magic, it's a second, differently-framed chance for the model to notice its own mistake. It won't catch everything, and on a model that already got it right, there's nothing to fix.

**This is the exact same pattern**, done four times in a row with much more structure, behind this lab's real research pipeline — see the very last cell of this notebook.

---
## Part 6 — Four Ways Prompts Go Wrong (Live Examples)

Bolstering the table from the slides — here's each failure mode actually happening, plus the fix, back to back.

### 6a. Format Drift
Asked for JSON, got prose with JSON buried somewhere inside it — or no JSON at all.

In [ ]:
weak_format_prompt = "List 3 pros and 3 cons of remote work as JSON."

strong_format_prompt = """List 3 pros and 3 cons of remote work.
Respond with ONLY JSON, no other text before or after, in exactly this shape:
{"pros": ["...", "...", "..."], "cons": ["...", "...", "..."]}"""

weak_out = ask(weak_format_prompt)
strong_out = ask(strong_format_prompt)
show("Weak format instruction", weak_out)
show("Strong format instruction (exact shape shown)", strong_out)

for label, out in [("weak", weak_out), ("strong", strong_out)]:
    try:
        json.loads(out.strip())
        print(f"{label}: valid JSON \u2713")
    except Exception:
        print(f"{label}: NOT valid JSON \u2717 — a script trying to json.loads() this would crash")

### 6b. Sycophancy
Model agrees with a wrong claim you stated confidently, instead of correcting you.

In [ ]:
leading_prompt = """The Great Wall of China is visible from space with the naked eye, right? \
Confirm this for my essay."""

neutral_prompt = """Is the Great Wall of China visible from space with the naked eye? \
Answer based on evidence, and say so directly if the common claim is false."""

show("Leading / confident framing", ask(leading_prompt))
show("Neutral, asked to verify independently", ask(neutral_prompt))

(The claim is false — it's a persistent myth. Watch whether the leading version pushes back or just goes along with you.)

### 6c. Instruction Drop
Long prompt, model follows only some of the instructions.

In [ ]:
buried_instructions = """Write a short paragraph about coffee. Make it interesting and \
engaging for readers. Also please keep in mind that some readers may not drink coffee \
themselves so try to be inclusive of that. It would also be good to mention some history \
if you can fit it in naturally without making it feel forced. Keep the tone light. \
Try to make it exactly 3 sentences. Don't use the word "beverage". End with a question \
for the reader."""

numbered_instructions = """Write a paragraph about coffee. Follow these rules exactly:
1. Exactly 3 sentences.
2. Include one historical fact.
3. Do not use the word "beverage".
4. End with a question directed at the reader."""

show("Buried in prose", ask(buried_instructions))
show("Numbered, explicit", ask(numbered_instructions))

**Count by hand**: how many of the 6 buried instructions actually got followed, vs. how many of the 4 numbered ones? Sentence count and the banned word are the easiest to check yourself.

### 6d. Ambiguity
Model picks a reasonable-but-wrong interpretation of a vague ask.

In [ ]:
ambiguous_prompt = "Make it shorter."   # shorter than WHAT? never specified

clarified_prompt = """Here is a paragraph:
"Remote work has transformed how companies operate. Employees can now work from \
anywhere, which has increased flexibility but also created new challenges around \
communication and team cohesion that managers are still learning to navigate."

Make this paragraph shorter: cut it to one sentence, keep the core claim about flexibility."""

show("Ambiguous (no referent given)", ask(ambiguous_prompt))
show("Clarified (referent + criterion given)", ask(clarified_prompt))

**Notice**: the ambiguous version has nothing to shorten — there was no prior text in this conversation for it to act on, so the model has to invent something to be "helpful." This is the most common real-world version of ambiguity: a follow-up instruction that assumes shared context the model doesn't actually have.

---
## Part 7 — Try It Yourself

Pick 2 of the 4 failure modes above and try to *deliberately* trigger them with your own prompt, then fix it. Write one sentence per failure mode: what went wrong, and what one change fixed it.

In [ ]:
# TODO: your own broken prompt, then your own fixed version
my_broken_prompt = "..."
my_fixed_prompt = "..."

# show("My broken version", ask(my_broken_prompt))
# show("My fixed version", ask(my_fixed_prompt))

---
## Where This Shows Up at Research Scale

Everything above was one small model, one or two calls, generic examples. This lab's actual research pipeline runs the *same* zero-shot / few-shot / chain-of-thought / reflection ideas on real HIPAA compliance scenarios, at much larger scale — and the results are not always intuitive. On Qwen2.5:72B (a much larger model than what you just used):

| Configuration | Accuracy (n=137 real scenarios) |
|---|---|
| Single-pass (no retrieval) | 92.7% |
| + retrieval added, no reflection | 84.7% — *worse* |
| + reflection (4-pass, like Part 5 but structured) | 92.7% — back to baseline |

Adding retrieval alone made things **worse** on this model, and reflection's real job turned out to be undoing that damage, not adding new capability. That's the opposite of what most people would guess. You can't predict this from theory — you have to run the ablation and look. This is exactly the kind of finding `slides-agentic-systems.html` walks through in full, with the actual code and row-by-row evidence.